In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath(".."))

import hashlib
import secrets
import numpy as np
import matplotlib.pyplot as plt

from ecc import *

# Módulo 3: Operaciones de Puntos

Aquí es donde la estructura algebraica del Módulo 1 se encuentra con las curvas del
Módulo 2. Necesitamos definir una operación que combine dos puntos de la curva y
produzca un tercer punto de la curva — una operación de grupo.

## 3.1 ¿Por qué no simplemente sumar las coordenadas?

El primer instinto: si $P = (x_1, y_1)$ y $Q = (x_2, y_2)$, ¿por qué no definir
$P + Q = (x_1 + x_2,\; y_1 + y_2)$?

Porque el resultado casi con certeza **no está en la curva**. Tanto $P$ como $Q$
satisfacen $y^2 = x^3 + 7$, pero $(y_1 + y_2)^2 \neq (x_1 + x_2)^3 + 7$ en
general. Has salido del conjunto. La cerradura se rompió. No hay grupo. No hay
esquema de firmas.

Así que la pregunta es: **¿qué operación sobre dos puntos de la curva garantiza
producir otro punto de la curva?**

La respuesta viene de la geometría de la curva misma. Una línea a través de dos
puntos en una curva cúbica (grado 3) siempre golpea la curva en exactamente un
punto más — esto es una consecuencia del teorema de Bezout. "Traza una línea,
encuentra la tercera intersección" es la única operación natural que **permanece en
la curva por construcción**.

El paso de reflexión (negar la coordenada $y$ de la tercera intersección) es lo
que hace que la operación satisfaga asociatividad y conmutatividad — convirtiéndola
en un grupo Abeliano propio.

Así que la "suma de puntos" no es sumar números. Es una construcción geométrica
que resulta obedecer las mismas reglas abstractas que la suma regular.

## 3.2 Intuición Geométrica

```
Suma de Puntos P + Q:          Duplicación de Punto 2P:

     ·  Q                           ·
    / \                            /|\
   /   \                          / | \
  P     \  ← línea secante      P  | línea tangente
   \     \                        \ |
    \     T                        \T
     \   /                          |
      \ /                           |
       R = P + Q  (reflejar T)      R = 2P  (reflejar T)
```

1. Traza una línea a través de P y Q (o la tangente en P para duplicar)
2. La línea golpea la curva en un tercer punto T
3. Refleja T sobre el eje x para obtener R = P + Q

## 3.2 Las Fórmulas

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

a_curve, b_curve = 1, 1

def curve_y(x):
    rhs = x**3 + a_curve * x + b_curve
    mask = rhs >= 0
    y = np.full_like(x, np.nan)
    y[mask] = np.sqrt(rhs[mask])
    return y

def find_third_point(px, py, qx, qy):
    """La recta por P y Q intersecta y^2 = x^3 + ax + b en un tercer punto."""
    if px == qx:
        return None, None
    lam = (qy - py) / (qx - px)
    nu = py - lam * px
    # x^3 - lam^2 * x^2 + ... = 0, roots sum to lam^2
    rx = lam**2 - px - qx
    ry = lam * rx + nu
    return rx, ry

# --- Dos ejemplos: Suma y Duplicación ---
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))

x_vals = np.linspace(-1.5, 4, 2000)
y_pos = curve_y(x_vals)

for ax in (ax1, ax2):
    ax.plot(x_vals, y_pos, 'b-', linewidth=2)
    ax.plot(x_vals, -y_pos, 'b-', linewidth=2)
    ax.axhline(0, color='gray', linewidth=0.3)
    ax.set_xlabel('x', fontsize=12)
    ax.set_ylabel('y', fontsize=12)
    ax.grid(True, alpha=0.2)

# --- Left panel: P + Q ---
Px, Py = 0.0, 1.0
Qx, Qy = 1.0, np.sqrt(1 + 1 + 1)
Tx, Ty = find_third_point(Px, Py, Qx, Qy)
Rx, Ry = Tx, -Ty

lam = (Qy - Py) / (Qx - Px)
nu = Py - lam * Px
x_line = np.linspace(-1.5, Tx + 0.5, 200)
y_line = lam * x_line + nu

ax1.plot(x_line, y_line, 'g--', linewidth=1.5, alpha=0.7, label='recta secante')
ax1.plot([Tx, Rx], [Ty, Ry], 'r:', linewidth=1.5, alpha=0.7, label='reflexión')
ax1.plot(Px, Py, 'ko', markersize=10, zorder=10)
ax1.plot(Qx, Qy, 'ko', markersize=10, zorder=10)
ax1.plot(Tx, Ty, 's', color='gray', markersize=9, zorder=10)
ax1.plot(Rx, Ry, 'r*', markersize=15, zorder=10)
ax1.annotate('P', (Px, Py), textcoords="offset points", xytext=(-15, 8), fontsize=13, fontweight='bold')
ax1.annotate('Q', (Qx, Qy), textcoords="offset points", xytext=(8, 8), fontsize=13, fontweight='bold')
ax1.annotate('T', (Tx, Ty), textcoords="offset points", xytext=(8, -15), fontsize=13, color='gray')
ax1.annotate('R = P + Q', (Rx, Ry), textcoords="offset points", xytext=(8, 8), fontsize=13, fontweight='bold', color='red')
ax1.set_title('Suma de Puntos: $P + Q$', fontsize=14)
ax1.set_xlim(-1.5, 4)
ax1.set_ylim(-5, 5)
ax1.legend(fontsize=10)

# --- Right panel: 2P (doubling via tangent) ---
Dx = 1.0
Dy = np.sqrt(Dx**3 + a_curve * Dx + b_curve)
# tangent slope: dy/dx from implicit differentiation: (3x^2 + a) / (2y)
lam_d = (3 * Dx**2 + a_curve) / (2 * Dy)
nu_d = Dy - lam_d * Dx
Tx2 = lam_d**2 - 2 * Dx
Ty2 = lam_d * Tx2 + nu_d
Rx2, Ry2 = Tx2, -Ty2

x_line2 = np.linspace(-1.5, Tx2 + 0.5, 200)
y_line2 = lam_d * x_line2 + nu_d

ax2.plot(x_line2, y_line2, 'g--', linewidth=1.5, alpha=0.7, label='recta tangente')
ax2.plot([Tx2, Rx2], [Ty2, Ry2], 'r:', linewidth=1.5, alpha=0.7, label='reflexión')
ax2.plot(Dx, Dy, 'ko', markersize=10, zorder=10)
ax2.plot(Tx2, Ty2, 's', color='gray', markersize=9, zorder=10)
ax2.plot(Rx2, Ry2, 'r*', markersize=15, zorder=10)
ax2.annotate('P', (Dx, Dy), textcoords="offset points", xytext=(-15, 8), fontsize=13, fontweight='bold')
ax2.annotate('T', (Tx2, Ty2), textcoords="offset points", xytext=(8, -15), fontsize=13, color='gray')
ax2.annotate('R = 2P', (Rx2, Ry2), textcoords="offset points", xytext=(8, 8), fontsize=13, fontweight='bold', color='red')
ax2.set_title('Duplicación de Punto: $2P$', fontsize=14)
ax2.set_xlim(-1.5, 4)
ax2.set_ylim(-5, 5)
ax2.legend(fontsize=10)

plt.tight_layout()
plt.show()

print("Izquierda: La recta por P y Q corta la curva en T. Reflejar T → R = P + Q.")
print("Derecha:   La tangente en P corta la curva en T. Reflejar T → R = 2P.")

In [ ]:
from typing import Optional

class Point:
    """Un punto en secp256k1 (o infinito)."""
    def __init__(self, x: Optional[int] = None, y: Optional[int] = None):
        self.x = x
        self.y = y
    
    def is_infinity(self) -> bool:
        return self.x is None or self.y is None
    
    def copy(self) -> 'Point':
        return Point(self.x, self.y)
    
    def __eq__(self, other):
        if self.is_infinity() and other.is_infinity():
            return True
        return self.x == other.x and self.y == other.y
    
    def __repr__(self):
        if self.is_infinity():
            return "O (punto en el infinito)"
        return f"({hex(self.x)[:16]}..., {hex(self.y)[:16]}...)"

G = Point(SECP_GX, SECP_GY)
INFINITY = Point()  # The identity element

print(f"Generador G:     {G}")
print(f"Identidad O:      {INFINITY}")

In [ ]:
def mod_inverse(a: int, p: int) -> int:
    """a⁻¹ mod p mediante el pequeño teorema de Fermat: a^(p-2) mod p."""
    return pow(a, p - 2, p)

def mod_sqrt(a: int, p: int) -> int:
    """√a mod p para p ≡ 3 (mod 4): a^((p+1)/4) mod p."""
    return pow(a, (p + 1) // 4, p)

def point_add(p1: Point, p2: Point) -> Point:
    """
    Suma dos puntos distintos en secp256k1.
    
    Pendiente: λ = (y₂ - y₁) / (x₂ - x₁) mod P
    Resultado: x₃ = λ² - x₁ - x₂
            y₃ = λ(x₁ - x₃) - y₁
    """
    if p1.is_infinity():
        return p2.copy()
    if p2.is_infinity():
        return p1.copy()
    
    if p1.x == p2.x:
        if (p1.y + p2.y) % SECP_P == 0:
            return Point()  # P + (-P) = O
        return point_double(p1)
    
    lam = ((p2.y - p1.y) * mod_inverse(p2.x - p1.x, SECP_P)) % SECP_P
    x3 = (lam * lam - p1.x - p2.x) % SECP_P
    y3 = (lam * (p1.x - x3) - p1.y) % SECP_P
    return Point(x3, y3)

def point_double(p: Point) -> Point:
    """
    Duplica un punto en secp256k1.
    
    Pendiente: λ = 3x² / 2y mod P  (tangente a la curva en P)
    Resultado: x₃ = λ² - 2x
            y₃ = λ(x - x₃) - y
    """
    if p.is_infinity() or p.y == 0:
        return Point()
    
    lam = (3 * p.x * p.x * mod_inverse(2 * p.y, SECP_P)) % SECP_P
    x3 = (lam * lam - 2 * p.x) % SECP_P
    y3 = (lam * (p.x - x3) - p.y) % SECP_P
    return Point(x3, y3)

def point_negate(p: Point) -> Point:
    """Negación: -P = (x, -y mod P)."""
    if p.is_infinity():
        return Point()
    return Point(p.x, (SECP_P - p.y) % SECP_P)

# Verify G is on the curve
lhs = (G.y * G.y) % SECP_P
rhs = (G.x ** 3 + 7) % SECP_P
print(f"¿G en la curva? y² mod P == x³+7 mod P: {lhs == rhs}  ✓")

# Verify group properties
G2 = point_double(G)
print(f"\n2G = {G2}")
print(f"G + O = G? {point_add(G, INFINITY) == G}  ✓  (identidad)")
neg_G = point_negate(G)
print(f"G + (-G) = O? {point_add(G, neg_G).is_infinity()}  ✓  (inverso)")

## 3.3 Multiplicación Escalar (Doblar y Sumar)

**Dónde estamos:** Podemos sumar dos puntos y duplicar un punto. Eso significa que
podemos calcular $G + G = 2G$, luego $2G + G = 3G$, luego $4G$, $5G$, y así
sucesivamente. Repetir la suma de puntos $d$ veces nos da $d \times G$ — y esa es
exactamente la función unidireccional de la introducción: **clave privada $d$
entra, clave pública $P$ sale.**

$$P = d \times G$$

Este es el momento en que ocurre la función unidireccional. Hacia adelante
(calcular $P$ a partir de $d$) es rápido. Hacia atrás (encontrar $d$ a partir
de $P$) es el problema del logaritmo discreto — inviable en secp256k1.

Pero $d$ es un número de 256 bits — hasta $\approx 10^{77}$. Sumar $G$ a sí
mismo tantas veces tomaría más que la edad del universo. Necesitamos un atajo.

**Doblar y sumar** usa la representación binaria de $d$ para hacerlo en
$O(\log d)$ operaciones — como máximo 256 duplicaciones y 256 sumas en lugar
de $10^{77}$ sumas.

```
Ejemplo: 13 × P  (13 = 1101 en binario)

Paso  Binario  Acción              Resultado
─────────────────────────────────────────
  0   1       resultado += sumando  P
      ─       sumando = 2×sumando   2P
  1   0       (saltar suma)         P
      ─       sumando = 2×sumando   4P
  2   1       resultado += sumando  P + 4P = 5P
      ─       sumando = 2×sumando   8P
  3   1       resultado += sumando  5P + 8P = 13P
```

In [ ]:
def scalar_mult(k: int, p: Point) -> Point:
    """Calcula k × P usando doblar y sumar. O(log k) operaciones."""
    if k == 0 or p.is_infinity():
        return Point()
    k = k % SECP_N
    if k == 0:
        return Point()
    
    result = Point()  # Start at O (identity)
    addend = p.copy()
    
    while k > 0:
        if k & 1:
            result = point_add(result, addend)
        addend = point_double(addend)
        k >>= 1
    
    return result

# Verify: 3G computed two ways
G3_algo = scalar_mult(3, G)
G3_manual = point_add(G, point_double(G))
print(f"3G (doblar y sumar): {G3_algo}")
print(f"3G (G + 2G):         {G3_manual}")
print(f"Coincide: {G3_algo == G3_manual}  ✓")

# The fundamental property: N × G = O (wraps around)
# (Don't actually compute this — it would take forever)
# But we can verify: (N-1)×G + G = O
print(f"\nFundamental: N × G = O (punto en el infinito)")
print(f"Esto significa que el espacio de claves privadas es cíclico con orden N.")
print(f"N ≈ 1.16 × 10⁷⁷ — más que los átomos en el universo observable.")

## 3.4 Claves Públicas Comprimidas

**Dónde estamos:** Ahora podemos calcular $P = d \times G$ — una clave pública a
partir de una clave privada. La clave pública $P$ es un punto $(x, y)$ en
secp256k1. Tanto $x$ como $y$ son números de 256 bits, así que almacenar ambos
toma 64 bytes (más un byte de prefijo = 65 bytes). Cada transacción de Bitcoin
incluye la clave pública, así que esto se acumula.

**El truco:** ¿Recuerdas la simetría vertical del Módulo 2? Para cualquier $x$,
la ecuación de la curva $y^2 = x^3 + 7$ tiene como máximo **dos** soluciones:
$y$ y $p - y$ — una par, una impar. Así que si conoces $x$ y cuál de los dos
valores de $y$ es (par o impar), puedes reconstruir el punto completo. Eso
reduce la clave pública casi a la mitad:

```
Sin comprimir: 65 bytes  [04 || x (32 bytes) || y (32 bytes)]
Comprimida:    33 bytes  [02/03 || x (32 bytes)]
                         02 = y par,  03 = y impar
```

Por eso cada dirección de Bitcoin que ves usa claves comprimidas — misma
seguridad, la mitad del espacio en la blockchain.

In [ ]:
def serialize_compressed(p: Point) -> bytes:
    """Punto → clave pública comprimida de 33 bytes."""
    prefix = 0x03 if (p.y & 1) else 0x02
    return bytes([prefix]) + p.x.to_bytes(32, 'big')

def parse_compressed(data: bytes) -> Point:
    """Clave pública comprimida de 33 bytes → Punto."""
    x = int.from_bytes(data[1:], 'big')
    y2 = (pow(x, 3, SECP_P) + 7) % SECP_P
    y = mod_sqrt(y2, SECP_P)
    if (y & 1) != (data[0] == 0x03):
        y = SECP_P - y
    return Point(x, y)

# Demo: generate a key pair
import secrets
private_key = secrets.randbelow(SECP_N - 1) + 1
public_key = scalar_mult(private_key, G)
compressed = serialize_compressed(public_key)

print(f"=== Generación de Par de Claves ===")
print(f"Clave privada (d):  {hex(private_key)[:20]}...")
print(f"Clave pública (P = d×G):")
print(f"  x: {hex(public_key.x)}")
print(f"  y: {hex(public_key.y)}")
print(f"Comprimida: {compressed.hex()}")
print(f"  Prefijo 0x{compressed[0]:02x} → y es {'impar' if compressed[0] == 0x03 else 'par'}")

# Round-trip verification
recovered = parse_compressed(compressed)
print(f"\nIda y vuelta: {public_key == recovered}  ✓")